# Eventbrite Ingestion Testing

**Source**: Eventbrite API v3  
**Type**: REST API — requires `EVENTBRITE_API_KEY`  
**Description**: Eventbrite is a leading self-service event management and ticketing platform covering a broad range of event types from conferences to concerts to community meetups. This notebook tests the Eventbrite ingestion pipeline end-to-end.

## What this notebook does
1. Initialises the environment and verifies the `EVENTBRITE_API_KEY` env var
2. Creates the Eventbrite pipeline via `PipelineFactory`
3. Executes the pipeline with a limited scope (max 10 events, 1 page)
4. Inspects normalised `EventSchema` results
5. Explores `custom_fields` specific to Eventbrite
6. Saves the batch to `data/batches/eventbrite/`
7. Produces a DataFrame summary

In [ ]:
import sys
import os
import logging
from pathlib import Path

# Add services/api to path so src.* imports resolve
REPO_ROOT = Path("__file__").resolve().parents[1]
API_ROOT = REPO_ROOT / "services" / "api"
if str(API_ROOT) not in sys.path:
    sys.path.insert(0, str(API_ROOT))

# Load .env from services/api
try:
    from dotenv import load_dotenv
    load_dotenv(API_ROOT / ".env")
    print("dotenv loaded")
except ImportError:
    print("python-dotenv not installed — skipping .env load")

# Logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(name)s: %(message)s",
)
logger = logging.getLogger("notebook.eventbrite")

# Check required API key
api_key = os.environ.get("EVENTBRITE_API_KEY")
if api_key:
    print(f"EVENTBRITE_API_KEY: set (length={len(api_key)})")
else:
    print("WARNING: EVENTBRITE_API_KEY is not set — pipeline execution will fail.")
    print("Set it in services/api/.env or export it in your shell.")
print(f"API_ROOT: {API_ROOT}")

In [ ]:
from src.ingestion.factory import PipelineFactory

factory = PipelineFactory()

# Temporarily enable the source for testing (disabled by default in YAML)
source_cfg = factory.get_source_config("eventbrite")
source_cfg["enabled"] = True

pipeline = factory.create_pipeline("eventbrite")
print(f"Pipeline created: {pipeline.__class__.__name__}")
print(f"Source type: {pipeline.source_type}")
print(f"Source name: {pipeline.config.source_name}")

In [ ]:
import asyncio

# Execute with limited scope: 1 page, 10 events
result = await pipeline.execute(
    max_pages=1,
    page_size=10,
)

print(f"Status        : {result.status}")
print(f"Execution ID  : {result.execution_id}")
print(f"Total fetched : {result.total_events_processed}")
print(f"Successful    : {result.successful_events}")
print(f"Failed        : {result.failed_events}")
print(f"Duration      : {result.duration_seconds:.2f}s")
if result.errors:
    print(f"Errors        : {result.errors}")

In [ ]:
print(f"Inspecting {len(result.events)} normalised events\n")
print("-" * 70)

for i, event in enumerate(result.events):
    venue = event.location.venue_name or "N/A"
    city  = event.location.city or "N/A"
    start = event.start_datetime.isoformat() if event.start_datetime else "N/A"
    etype = event.event_type.value if event.event_type else "N/A"
    cf_keys = list(event.custom_fields.keys()) if event.custom_fields else []

    print(f"[{i+1:02d}] {event.title}")
    print(f"     Venue  : {venue} ({city})")
    print(f"     Date   : {start}")
    print(f"     Type   : {etype}")
    print(f"     CF keys: {cf_keys}")
    print()

In [ ]:
print("custom_fields inspection for Eventbrite\n")
print("-" * 70)

for i, event in enumerate(result.events[:3]):
    print(f"Event: {event.title}")
    if event.custom_fields:
        for key, val in event.custom_fields.items():
            print(f"  {key}: {val}")
    else:
        print("  (no custom_fields)")
    print()

# Aggregate all custom_field keys across all events
all_cf_keys = set()
for event in result.events:
    if event.custom_fields:
        all_cf_keys.update(event.custom_fields.keys())
print(f"All custom_fields keys found: {sorted(all_cf_keys)}")

In [ ]:
from src.ingestion.batch_writer import BatchWriter

writer = BatchWriter()

batch_path = writer.save_batch(
    source_name="eventbrite",
    events=result.events,
    batch_id=result.execution_id,
    metadata={
        "pipeline_status": result.status.value,
        "duration_seconds": result.duration_seconds,
    },
)

print(f"Batch saved to : {batch_path}")
print(f"File size      : {batch_path.stat().st_size:,} bytes")

In [ ]:
import pandas as pd

if result.events:
    df = pipeline.to_dataframe(result.events)
    print(f"DataFrame shape: {df.shape}")
    print(f"\nColumns: {list(df.columns)}")
    print("\nBasic stats:")
    print(f"  Events          : {len(df)}")
    print(f"  Unique cities   : {df['city'].nunique()}")
    print(f"  Avg quality     : {df['data_quality_score'].mean():.2f}")
    print(f"  Free events     : {df['price_is_free'].sum()}")
    print("\nSample (title, city, start_datetime, data_quality_score):")
    print(df[["title", "city", "start_datetime", "data_quality_score"]].to_string(index=False))
else:
    print("No events to display.")

In [ ]:
await pipeline.close()
print("Pipeline closed. Resources released.")